# Finetuning - QLoRA

https://huggingface.co/docs/peft/en/developer_guides/quantization


## Quantization
4비트 양자화(4-bit Quantization)는 모델의 가중치를 정밀도가 낮은 4비트 데이터 형식으로 변환하여 메모리 사용량을 획기적으로 줄이는 기술이다. 지적한 대로 모든 파라미터가 양자화 대상이 되는 것은 아니며, 성능 유지를 위해 전략적으로 적용된다.
4비트 양자화는 **"대세인 가중치는 작게 줄이고, 민감한 레이어와 통계 정보는 원본을 유지"**하는 전략이다. 이를 통해 일반 소비자용 GPU(예: RTX 3090/4090 24GB)에서도 30B급 대형 모델을 구동할 수 있게 된다.


**1. 4비트 양자화 시 메모리 변화**

30B 파라미터 모델을 기준으로 계산하면 다음과 같은 변화가 발생한다.

- **BF16 (기본):** 파라미터당 2바이트 $\rightarrow$ 약 **60GB** 필요
- **4-bit (양자화):** 파라미터당 0.5바이트 $\rightarrow$ 약 **15GB** 필요 (이론상 1/4 수준)

실제로는 양자화 과정에서 발생하는 스케일링 계수(Scaling Factor)와 메타데이터 때문에 약 **17~18GB** 정도의 VRAM을 사용하게 된다.

**2. 왜 모든 파라미터를 양자화하지 않는가?**

모델의 성능(Perplexity) 저하를 최소화하기 위해 **혼합 정밀도(Mixed Precision)** 방식을 사용한다.

- **양자화 대상 (Linear Layers):** 모델의 대부분을 차지하는 행렬 연산 가중치(Attention, MLP 레이어 등)는 4비트로 변환하여 용량을 줄인다.
- **양자화 제외 (Sensitive Layers):**
    - **Normalization 레이어:** LayerNorm 등은 수치 민감도가 매우 높아 원본 정밀도(FP32/BF16)를 유지한다.
    - **Embedding 레이어:** 텍스트를 벡터로 변환하는 첫 단계이므로 정밀도가 중요하다.
    - **LM Head:** 최종 출력층은 예측 정확도를 위해 보통 양자화하지 않는다.

**3. 주요 양자화 기법 (NF4)**

단순히 소수점을 자르는 것이 아니라, 데이터의 분포를 고려한 알고리즘을 사용한다. 가장 대표적인 것이 **NF4(NormalFloat 4)**이다.

- **특징:** 가중치가 정규분포를 따른다는 가정하에, 값이 몰려 있는 구간에는 촘촘하게, 값이 적은 구간에는 넓게 비트를 할당한다.
- **장점:** 일반적인 4비트 정수형(Int4)보다 정보 손실이 훨씬 적어 모델의 추론 능력을 잘 보존한다.

In [1]:
%pip install -Uq transformers datasets accelerate trl peft bitsandbytes hf_transfer wandb

Note: you may need to restart the kernel to use updated packages.


In [2]:
!nvidia-smi # GPU 확인 (모델명 / GPU ID / 메모리 사용량)

Wed Sep  9 06:28:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    On  |   00000000:C1:00.0 Off |                    0 |
| N/A   24C    P8             36W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
# 로컬 기준 환경변수 설정
# from dotenv import load_dotenv
# import os

# load_dotenv()
# os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
# os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN')

In [4]:
# Runpod 기준 환경변수 설정(Pod 에 환경 변수가 있어야 함)
import os

HF_TOKEN = os.environ['HF_TOKEN'] 

## 데이터셋 로드
https://huggingface.co/datasets/capybaraOh/naver-economy-news2stock

In [5]:
from datasets import load_dataset # HuggingFace 데이터셋 로더

# HuggingFace Hub train split 로드
dataset = load_dataset('capybaraOh/naver-economy-news2stock', split='train')
print(len(dataset))
dataset # Dataset 객체 정보

README.md:   0%|          | 0.00/347 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.27MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

1000


Dataset({
    features: ['system', 'user', 'assistant'],
    num_rows: 1000
})

In [6]:
dataset[0] # {'system' : ... , 'user' : ... , 'assistatnt':...}

{'system': "\n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n",
 'user': '추경호 중기 수출지원 총력 무역금융 40조 확대\n앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대

In [7]:
# HuggingFace Dataset을 학습 / 평가 분리 후 Chat 메시지 포맷으로 변환
test_ratio = 0.2 # 평가셋 비율

train_data = [] # 학습 데이터 리스트
test_data = []  # 평가 데이터 리스트

data_indices = list(range(len(dataset))) # 전체 인덱스
test_size = int(len(dataset) * test_ratio) # 평가셋 크기

test_data_indices = data_indices[:test_size] # 앞부분은 평가셋 (인덱스)
train_data_indices = data_indices[test_size:] # 나머지는 학습셋 (인덱스)

# OpenAI / Chat 학습용 messages 포맷 : {'messages' : [{system},{user},{assistant}]}
def format_data(data):
    return {
        'messages' : [
            {
                'role' : 'system',
                'content' : data['system']
            },
            {
                'role' : 'user',
                'content' : data['user']
            },
            {
                'role' : 'assistant',
                'content' : data['assistant']
            },
        ]
    }

train_data = [format_data(dataset[i]) for i in train_data_indices] # 학습 인덱스 -> 학습 dict
test_data = [format_data(dataset[i]) for i in test_data_indices] # 평가 인덱스 -> 평가 dict

print(len(train_data))
print(len(test_data))

800
200


In [8]:
train_data[256] # message 포맷 dict 확인

{'messages': [{'role': 'system',
   'content': "\n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n"},
  {'role': 'user',
   'content': '국토부 세종시 부적격 당첨자 철저히 조사해 엄중 조치\n원희룡 계약취소 및 주택환수 형사고발까지 필요 조치 취할 것 원희룡 국토교통부 장관. 2022.7.6 뉴스1 © News1 김명섭 기자 서울 뉴스1 박승희 기자 국토교통부는 감사원 감사에서 세종시 무자격 특공 당첨자가 무더기로 적발된 것과 관련해 위법행위 여부를 철저히 조사해 위법성이 밝혀지는 경우 엄중히 조치할 것 이라고 6일 밝혔다. 감사원은 세종시 이전기관 종사자 주택 특별공급에 대한 감사를 진행한 결과 총 45건 76명 에 대한 감사 결과를 확정했다. Δ대상관리 부실 Δ확인서 부당 발급 Δ확인서 위조 Δ중복 당첨 Δ지도·감독 소홀 등 사유로 무자격자가 대거 당첨된 것으로 드러났다. 국토부는 감사원으로부터 결과를 통보 받았으며 이에 따른 

In [9]:
# List -> HuggingFace Dataset (내용은 그대로, 컨테이너만 변경)
from datasets import Dataset

train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

train_dataset[256]

{'messages': [{'role': 'system',
   'content': "\n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n"},
  {'role': 'user',
   'content': '국토부 세종시 부적격 당첨자 철저히 조사해 엄중 조치\n원희룡 계약취소 및 주택환수 형사고발까지 필요 조치 취할 것 원희룡 국토교통부 장관. 2022.7.6 뉴스1 © News1 김명섭 기자 서울 뉴스1 박승희 기자 국토교통부는 감사원 감사에서 세종시 무자격 특공 당첨자가 무더기로 적발된 것과 관련해 위법행위 여부를 철저히 조사해 위법성이 밝혀지는 경우 엄중히 조치할 것 이라고 6일 밝혔다. 감사원은 세종시 이전기관 종사자 주택 특별공급에 대한 감사를 진행한 결과 총 45건 76명 에 대한 감사 결과를 확정했다. Δ대상관리 부실 Δ확인서 부당 발급 Δ확인서 위조 Δ중복 당첨 Δ지도·감독 소홀 등 사유로 무자격자가 대거 당첨된 것으로 드러났다. 국토부는 감사원으로부터 결과를 통보 받았으며 이에 따른 

## BaseModel + Quantizaton-Config

`BitsAndBytesConfig`는 Hugging Face Transformers에서 대형 모델을 8비트 또는 4비트로 양자화(quantization)하여 메모리 사용량을 줄이고, 저사양 환경에서도 대형 모델을 사용할 수 있게 도와주는 설정 클래스이다.

**주요 파라미터 목록**

| 파라미터명                  | 설명                                                                                          | 예시 값            |
|-----------------------------|----------------------------------------------------------------------------------------------|-------------------|
| `load_in_8bit`              | 8비트 양자화 활성화 여부. True로 설정 시 8비트로 모델 로드.                                    | True, False       |
| `load_in_4bit`              | 4비트 양자화 활성화 여부. True로 설정 시 4비트로 모델 로드.                                    | True, False       |
| `bnb_4bit_quant_type`       | 4비트 양자화 타입. `nf4`(NormalFloat4, 기본값), `fp4` 중 선택.                                 | "nf4", "fp4"      |
| `bnb_4bit_compute_dtype`    | 연산에 사용할 데이터 타입. 보통 `torch.float16`, `torch.bfloat16`, `torch.float32` 중 선택.     | torch.bfloat16    |
| `bnb_4bit_use_double_quant` | 이중 양자화 사용 여부. True로 설정 시 추가 양자화로 메모리 절감 가능.                           | True, False       |
| `llm_int8_threshold`        | 8비트 양자화 시 threshold 지정. 값이 낮을수록 더 많은 파라미터가 8비트로 변환됨.                | 0.0 ~ 6.0         |
| `llm_int8_skip_modules`     | 양자화에서 제외할 모듈 리스트.                                                                | ["lm_head"]       |
| `bnb_4bit_quant_storage`    | 4비트 파라미터 저장에 사용할 타입. 기본값은 `torch.uint8`.                                    | torch.uint8       |


- **load_in_8bit**  
  8비트 양자화를 활성화하는 플래그이다. True로 설정 시 모델 파라미터를 8비트 정수로 변환하여 메모리 사용량을 약 75%까지 줄일 수 있다.

- **load_in_4bit**  
  4비트 양자화를 활성화하는 플래그이다. True로 설정 시 더욱 극적인 메모리 절감 효과를 볼 수 있다. 4비트 양자화는 QLoRA 등 최신 연구에서 자주 사용된다.

- **bnb_4bit_quant_type**  
  4비트 양자화 시 사용할 데이터 타입을 지정한다.  
  - `nf4`: NormalFloat4 (기본값, QLoRA에서 주로 사용)  
  - `fp4`: FP4 타입.

- **bnb_4bit_compute_dtype**  
  연산(Forward/Backward) 시 사용할 데이터 타입을 지정한다.  
  - `torch.float16`, `torch.bfloat16`, `torch.float32` 등이 있다.  
  - 16비트 타입을 사용하면 연산 속도가 빨라지고, 메모리 사용량도 줄일 수 있다.

- **bnb_4bit_use_double_quant**  
  이중 양자화(nested quantization)를 활성화하는 옵션이다. True로 설정 시 한 번 더 양자화를 적용하여 메모리 사용량을 추가로 절감할 수 있다. 메모리 부족 시 유용하다.

- **llm_int8_threshold**  
  8비트 양자화 시 threshold 값을 조정하여, threshold 이하의 weight만 8비트로 변환한다. 값이 낮을수록 더 많은 파라미터가 8비트로 변환된다.

- **llm_int8_skip_modules**  
  양자화에서 제외할 모듈(레이어) 리스트를 지정한다. 예를 들어, 출력 레이어(`lm_head`) 등은 양자화에서 제외할 수 있다.

- **bnb_4bit_quant_storage**  
  4비트 파라미터 저장에 사용할 데이터 타입을 지정한다. 기본값은 `torch.uint8`이다.


**활용 팁**

- **메모리가 부족하다면**: `bnb_4bit_use_double_quant=True`로 설정.
- **정밀도가 중요하다면**: `bnb_4bit_quant_type="nf4"`로 설정.
- **학습 속도가 중요하다면**: `bnb_4bit_compute_dtype`를 16비트(float16, bfloat16)로 설정.

- `BitsAndBytesConfig`는 4비트/8비트 양자화 옵션을 통합 관리하며, 파라미터 조합을 통해 다양한 하드웨어 환경에 맞는 최적화가 가능하다.

In [10]:
# 4bit 양자화 설정(BitsAndBytesConfig)
from transformers import BitsAndBytesConfig
import torch

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # 4bit 양자화(모델 가중치를 4bit)
    bnb_4bit_quant_type = 'nf4',            # 4bit 양자화 방식
    bnb_4bit_use_double_quant=True,         # 이중 양자화 : 메모리 / 정확도 균형 개선
    bnb_4bit_compute_dtype=torch.bfloat16   # 연산 방식 bfloat16
)

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM  # 토크나이저 / 생성형 모델 자동 로더
import torch

pretrained_model_name = 'NCSOFT/Llama-VARCO-8B-Instruct'  # 사전학습 모델명

model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name,
    dtype = torch.bfloat16,  # 가중치 로딩 dtype(bf16)
    device_map = 'auto',      # 환경에 맞춰 CPU / GPU 자동 배치
    quantization_config = quant_config   # 4bit 양자화 설정 적용
)

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name)  # 해당 모델의 토크나이저 로드

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/51.2k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

## BaseModel + Quantizaton-Config

`BitsAndBytesConfig`는 Hugging Face Transformers에서 대형 모델을 8비트 또는 4비트로 양자화(quantization)하여 메모리 사용량을 줄이고, 저사양 환경에서도 대형 모델을 사용할 수 있게 도와주는 설정 클래스이다.

**주요 파라미터 목록**

| 파라미터명                  | 설명                                                                                          | 예시 값            |
|-----------------------------|----------------------------------------------------------------------------------------------|-------------------|
| `load_in_8bit`              | 8비트 양자화 활성화 여부. True로 설정 시 8비트로 모델 로드.                                    | True, False       |
| `load_in_4bit`              | 4비트 양자화 활성화 여부. True로 설정 시 4비트로 모델 로드.                                    | True, False       |
| `bnb_4bit_quant_type`       | 4비트 양자화 타입. `nf4`(NormalFloat4, 기본값), `fp4` 중 선택.                                 | "nf4", "fp4"      |
| `bnb_4bit_compute_dtype`    | 연산에 사용할 데이터 타입. 보통 `torch.float16`, `torch.bfloat16`, `torch.float32` 중 선택.     | torch.bfloat16    |
| `bnb_4bit_use_double_quant` | 이중 양자화 사용 여부. True로 설정 시 추가 양자화로 메모리 절감 가능.                           | True, False       |
| `llm_int8_threshold`        | 8비트 양자화 시 threshold 지정. 값이 낮을수록 더 많은 파라미터가 8비트로 변환됨.                | 0.0 ~ 6.0         |
| `llm_int8_skip_modules`     | 양자화에서 제외할 모듈 리스트.                                                                | ["lm_head"]       |
| `bnb_4bit_quant_storage`    | 4비트 파라미터 저장에 사용할 타입. 기본값은 `torch.uint8`.                                    | torch.uint8       |


- **load_in_8bit**  
  8비트 양자화를 활성화하는 플래그이다. True로 설정 시 모델 파라미터를 8비트 정수로 변환하여 메모리 사용량을 약 75%까지 줄일 수 있다.

- **load_in_4bit**  
  4비트 양자화를 활성화하는 플래그이다. True로 설정 시 더욱 극적인 메모리 절감 효과를 볼 수 있다. 4비트 양자화는 QLoRA 등 최신 연구에서 자주 사용된다.

- **bnb_4bit_quant_type**  
  4비트 양자화 시 사용할 데이터 타입을 지정한다.  
  - `nf4`: NormalFloat4 (기본값, QLoRA에서 주로 사용)  
  - `fp4`: FP4 타입.

- **bnb_4bit_compute_dtype**  
  연산(Forward/Backward) 시 사용할 데이터 타입을 지정한다.  
  - `torch.float16`, `torch.bfloat16`, `torch.float32` 등이 있다.  
  - 16비트 타입을 사용하면 연산 속도가 빨라지고, 메모리 사용량도 줄일 수 있다.

- **bnb_4bit_use_double_quant**  
  이중 양자화(nested quantization)를 활성화하는 옵션이다. True로 설정 시 한 번 더 양자화를 적용하여 메모리 사용량을 추가로 절감할 수 있다. 메모리 부족 시 유용하다.

- **llm_int8_threshold**  
  8비트 양자화 시 threshold 값을 조정하여, threshold 이하의 weight만 8비트로 변환한다. 값이 낮을수록 더 많은 파라미터가 8비트로 변환된다.

- **llm_int8_skip_modules**  
  양자화에서 제외할 모듈(레이어) 리스트를 지정한다. 예를 들어, 출력 레이어(`lm_head`) 등은 양자화에서 제외할 수 있다.

- **bnb_4bit_quant_storage**  
  4비트 파라미터 저장에 사용할 데이터 타입을 지정한다. 기본값은 `torch.uint8`이다.


**활용 팁**

- **메모리가 부족하다면**: `bnb_4bit_use_double_quant=True`로 설정.
- **정밀도가 중요하다면**: `bnb_4bit_quant_type="nf4"`로 설정.
- **학습 속도가 중요하다면**: `bnb_4bit_compute_dtype`를 16비트(float16, bfloat16)로 설정.

- `BitsAndBytesConfig`는 4비트/8비트 양자화 옵션을 통합 관리하며, 파라미터 조합을 통해 다양한 하드웨어 환경에 맞는 최적화가 가능하다.

In [12]:
# 하나의 샘플만 openai 방식 메시지 -> llama3 방식 메시지로 변환
text = tokenizer.apply_chat_template(train_dataset[128]['messages'], tokenize=False)
print(text)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,
특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.

다음 출력지시사항을 지켜주세요.
1. 뉴스와 종목간의 연관성을 발견할 수 없다면:
    - stock_related를 False로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
2. 뉴스와 종목간의 연관성을 발견했다면:
    - stock_related를 True로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.
    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.
    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.<|eot_id|><|start_header_id|>user<|end_header_id|>

에어부산 울란바토르·오사카 노선 재개
에어부산이 김해국제공항에서 출발하는 몽골 울란바토르와 일본 오사카 노선 운항을 각각 주 2회 일정으로 코로나19 팬데믹 사태 이후 28개월 만에 재개한다고 1일 밝혔다. 부산 울란바토르 노선은 김해국제공항에서 오전 8시 25분에 출발해 현지 공항에 오전 11시 40분 도착하고 귀국편은 오후 1시에 출발해 김해공항에 오후 5시 30분 도착하는 일정으로 주 2회 운항한다. 몽골은 입국 시 코로나19 검사와 백신 접종 여부를 확인하지 않아 자유롭게 여행이 가능한 국가다. 부산 오사카 노선은 김해공항에서 오전 8시 35분에 출발해 간사이공항에 오전 10시 도착 귀국편은 간사이공항에서 낮 12시에 출발해 김해공항에 오후 1시 30분 도착하는 일정으로 

In [13]:
# Lllam3 계열 Chat 학습용 데이터 콜레이터 : 프롬프트 생성 -> 토크나이즈/패딩 -> assistant 구간만 라벨링
# - 배치(messages)를 입력받아, 모델 학습 텐서로 변환
def data_collator(batch, tokenizer=tokenizer, max_length=8192):
    # 1. 프롬프트 생성
    prompts = []  # 배치 프롬프트 문자열을 담을 리스트
    for example in batch:
        prompt = '<|begin_of_text|>'  # 프롬프트 시작 토큰
        for msg in example['messages']:  # 샘플 내에서 (system/user/assistant) 순회
            role = msg['role']
            content = msg['content'].strip()
            # role + content로 템플릿을 완성
            prompt += f"<|start_header_id|>{role}<|end_header_id|>\n{content}<|eot_id|>"
        prompts.append(prompt)
    # print(prompts)

    # 2. 토큰처리 / 패딩 / 텐서 변환
    
    # 프롬프트를 토큰화해서 텐서로 변환
    tokenized = tokenizer(
        prompt,
        truncation = True,         # 최대 길이 초과시 자름
        max_length = max_length,   # 최대 길이 설정
        padding = True,            # 최대 길이 미만시 패딩 처리
        return_tensors = "pt"      # Pytorch Tensor 반환
    )
    input_ids = tokenized['input_ids']  # 토큰 id 텐서
    attention_mask = tokenized['attention_mask']  # 패딩 마스크 텐서
    # print(tokenized)
    # print(len(tokenized['input_ids'][0]))
    # print(len(tokenized['input_ids'][1]))
    # print(len(tokenized['attention_mask'][0]), len(tokenized['attention_mask'][0].sum().item()))
    # print(len(tokenized['attention_mask'][0]), len(tokenized['attention_mask'][0].sum().item()))

    # 3. 라벨 생성
    labels = torch.full_like(input_ids, fill_value=-100)  # input_ids shape으로 -100 기본값. (손실 계산 제외)
    print(labels.shape)

    assistant_header = '<|start_header_id|>assistant<|end_header_id|>\n'
    assistant_token_id = tokenizer.encode(assistant_header, add_special_tokens=False)  # 헤더의 토큰 패턴
    eot_token = '<|eot_id|>'
    eot_token_id = tokenizer.encode(eot_token, add_special_tokens=False)  # 종료 토큰의 토큰 패턴
    print(assistant_token_id)
    print(eot_token_id)

    for i, ids in enumerate(input_ids):  # 각 샘플별로 순회
        ids_list = ids.tolist()  # 슬라이싱 사용하기 위해 list로 변경
        
        # assistant 답변 시작위치 찾음
        # - <|start_header_id|>assistant<|end_header_id|>\n 다음 인덱스부터 답변으로 수집
        start = None  # 답변 시작 인덱스 초기화
        for idx in range(len(ids_list) - len(assistant_token_id) + 1):  # 헤더 길이만큼 탐색
            # assistant header 패턴과 매칭시
            if ids_list[idx: idx + len(assistant_token_id)] == assistant_token_id:
                start = idx + len(assistant_token_id)  # 헤더 다음 토큰부터 라벨링 시작
                break  # 첫 번째 assistant 구간 후 해당 for문 탈출
        
        # 답변 끝 위치 찾음 : <|eot_id|> 전까지
        if start is not None:  # assistant 헤더를 찾은 경우
            end = None  # 답변 종료 인덱스 초기화
            for idx in range(start, len(ids_list) - len(eot_token_id) + 1):  # start ~ 종료 토큰 전
                # start부터 종료 패턴과 매칭시
                if ids_list[idx: idx + len(eot_token_id)] == eot_token_id:
                    end = idx + len(eot_token_id)  # eot까지 포함한 구간 설정
                    break  # 첫 번째 eot 구간 후 해당 for문 탈출
        print(f"{i}: {start} ~ {end}")
        labels[i, start:end] = input_ids[i, start:end]  # assistant 답변 부분만 정답 라벨로 복사

    return {
        'input_ids': input_ids,  # 모델 입력
        'attention_mask': attention_mask,  # 패딩 마스크
        'labels': labels  # 손실 계산용 라벨(assistant 답변 부분만 labels 활용)
    }

data_collator([train_dataset[0], train_dataset[1]])

torch.Size([1, 1112])
[128006, 78191, 128007, 198]
[128009]
0: 623 ~ 1112


{'input_ids': tensor([[128000, 128006,   9125,  ...,  63466,     92, 128009]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]]),
 'labels': tensor([[  -100,   -100,   -100,  ...,  63466,     92, 128009]])}

### Causal Language Model 파인튜닝: input_ids와 labels 구조 이해

**데이터 구조**
```
input_ids:  [system_tokens..., user_tokens..., assistant_tokens...]  # 전체 시퀀스
labels:     [-100, -100, ..., -100, assistant_tokens...]          # assistant만 학습 대상
```

| 항목 | 내용 |
| --- | --- |
| **Input IDs** | 프롬프트 + 정답 (전체 시퀀스) |
| **Labels** | `-100` (프롬프트 구간) + 정답 토큰 (답변 구간) |
| **결과** | 모델은 입력을 다 보지만, 오직 답변을 맞히는 과정에서만 학습이 일어남 |




> **질문에 해당하는 input_ids에 이미 답이 포함되어 있다!**
>
> **"답이 이미 있는데 어떻게 학습하는가?"**
>
> 모델은 정답을 "보면서" 각 위치에서 올바른 다음 토큰을 예측하는 법을 배운다. 마치 학생이 모범답안을 보며 "이 상황에서는 이렇게 답해야 한다"를 학습하는 것과 같다. 이것이 현대 LLM 파인튜닝의 핵심 메커니즘이다!


**_1. 인과적 언어 모델링 (Causal Language Modeling):_**

LLM(Llama, GPT 등)은 **이전 토큰들을 보고 다음 토큰을 예측**하는 방식으로 학습한다. 따라서 학습 데이터에는 프롬프트와 정답이 모두 포함된 전체 문장이 들어가야 한다.

* **학습 원리:** 모델은 번째 토큰까지를 입력으로 받아 번째 토큰을 예측한다.
* **구조:** `input_ids`가 `[A, B, C, D]`라면, 모델은 내부적으로 `A`를 보고 `B`를, `A, B`를 보고 `C`를 예측하는 과정을 동시에 수행한다.

**_2. Teacher Forcing 기법:_**
```
Position:   [0, 1, 2, 3, 4, 5, 6, 7, 8]
input_ids:  [A, B, C, D, E, F, G, H, I]
labels:     [-100, -100, -100, -100, E, F, G, H, I]
```

학습 과정:
- Position 4: A,B,C,D를 보고 → E 예측
- Position 5: A,B,C,D,E를 보고 → F 예측  
- Position 6: A,B,C,D,E,F를 보고 → G 예측

**_3. Labels와 Loss 계산의 역할:_**

`input_ids`에 정답이 포함되어 있더라도, 모델이 모든 구간에 대해 학습(손실 계산)을 수행하는 것은 아니다. 이때 중요한 역할을 하는 것이 바로 코드에 작성된 **`labels`**이다.

* **-100의 의미:** PyTorch의 `CrossEntropyLoss`는 기본적으로 레이블 값이 `-100`인 위치를 무시(ignore)한다.
* **학습 차단:** 코드에서 프롬프트(User 질문 등) 구간의 레이블을 `-100`으로 설정했기 때문에, 모델이 프롬프트 내용을 예측하며 발생하는 오차는 학습에 반영되지 않는다.
* **학습 집중:** 오직 `assistant`의 답변 구간에 해당하는 `labels`만 실제 `input_ids` 값을 가지므로, 모델은 **"프롬프트가 주어졌을 때 정답을 생성하는 방법"**에 대해서만 가중치를 업데이트한다.


**학습 vs 추론의 차이**

**_학습 시:_**
```
input_ids: <system>당신은 금융분석가</system><user>뉴스내용</user><assistant>분석결과</assistant>
labels:    [-100, -100, ..., -100, 분석결과_토큰들]
```

**_추론 시:_**
```
input:  <system>당신은 금융분석가</system><user>뉴스내용</user><assistant>
output: 분석결과 (모델이 한 토큰씩 생성)
```

In [14]:
# 테스트 데이터로 변환된 결과 확인
example = train_dataset[256]
batch = data_collator([example]) # 배치 1개로 콜레이터 적용

print(f"{batch['input_ids'].shape}")    # input_ids 텐서 shape
print(f"{batch['attention_mask'].shape}") # attention_mask 텐서 shape
print(f"{batch['labels'].shape}") # labels 텐서 shape

torch.Size([1, 1093])
[128006, 78191, 128007, 198]
[128009]
0: 844 ~ 1093
torch.Size([1, 1093])
torch.Size([1, 1093])
torch.Size([1, 1093])


In [15]:
print(batch['input_ids'][0].tolist()) # 0번째 샘플 input_ids 리스트
print(batch['attention_mask'][0].tolist()) # 0번째 샘플 attention_mask 리스트
print(batch['labels'][0].tolist()) # 0번째 샘플 labels 리스트 (답변 제외 -100)

[128000, 128006, 9125, 128007, 198, 65895, 83628, 34804, 104193, 123061, 14, 127463, 111068, 120226, 125959, 102612, 96318, 27797, 18359, 87097, 103168, 34983, 114942, 101360, 345, 108159, 30381, 59134, 41953, 99458, 88708, 19954, 101412, 116129, 41871, 235, 30381, 14, 64189, 30381, 115754, 58126, 64189, 11, 111436, 11, 106589, 93292, 18918, 109862, 44005, 104193, 123061, 14, 127463, 109862, 116425, 20565, 80052, 382, 13447, 49531, 62226, 22035, 30426, 115790, 18359, 67890, 115061, 92769, 627, 16, 13, 111068, 25941, 81673, 99458, 88708, 63375, 21028, 78453, 101106, 111490, 121712, 48936, 29833, 47782, 115300, 512, 262, 482, 5708, 54356, 18918, 3641, 17835, 114839, 92245, 627, 262, 482, 12399, 19954, 111068, 120226, 87097, 103168, 18359, 114839, 92245, 627, 17, 13, 111068, 25941, 81673, 99458, 88708, 63375, 21028, 78453, 101106, 111490, 121712, 101528, 33390, 512, 262, 482, 5708, 54356, 18918, 3082, 17835, 114839, 92245, 627, 262, 482, 12399, 19954, 111068, 120226, 87097, 103168, 18359,

In [16]:
# labels 에서 -100 제거한 후, assistant 정답 구간만 디코딩
label_ids = [token_id for token_id in batch['labels'][0].tolist() if token_id != -100]
text = tokenizer.decode(label_ids)
text

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


'{"stock_related":false,"summary":"국토교통부가 감사원 감사에서 적발된 세종시 이전기관 종사자 특별공급 부적격 당첨자 76명에 대해 사실관계를 철저히 조사하고, 위법성이 확인되면 계약 취소·주택 환수·형사고발 등 강력한 조치를 취하겠다고 밝혔습니다. 감사 결과에는 무자격 당첨, 특별공급 확인서 부당 발급 및 위조, 중복 당첨, 관계기관의 관리·감독 소홀 등이 포함됐습니다. 또한 부적격 당첨자에게 주택을 공급한 사업자에 대해서도 법령 위반 여부를 조사해 고발이나 영업정지 등의 조치를 검토할 방침입니다. 이번 조치는 주택공급 시장의 불법·교란행위를 차단하고 특별공급 제도의 공정성과 관리·감독을 강화하려는 데 목적이 있습니다.","positive_stocks":[],"positive_keywords":[],"positive_reasons":"","negative_stocks":[],"negative_keywords":[],"negative_reasons":""}<|eot_id|>'

In [17]:
# input_ids 를 토큰/문자 단위로 디코딩해서 확인
text_tokens = [] # 토큰 id를 디코딩한 문자열을 담을 리스트
for i, token_id in enumerate(batch['input_ids'][0].tolist()): # 토큰 ID 순회
    decoded_str = tokenizer.decode([token_id]) # 토큰 1개를 문자열로 디코딩
    text_tokens.append(decoded_str) # 디코딩 결과 누적
text_tokens

['<|begin_of_text|>',
 '<|start_header_id|>',
 'system',
 '<|end_header_id|>',
 '\n',
 '당',
 '신',
 '은',
 ' 금',
 '융',
 '/',
 '경제',
 ' 뉴',
 '스의',
 ' 핵',
 '심',
 '내',
 '용',
 '을',
 ' 요',
 '약',
 '해',
 ' 설명',
 '하고',
 ',\n',
 '특',
 '정',
 ' 상',
 '장',
 ' 종',
 '목',
 '에',
 ' 미',
 '치는',
 ' �',
 '�',
 '정',
 '/',
 '부',
 '정',
 ' 영향',
 '여',
 '부',
 ',',
 ' 이유',
 ',',
 ' 근',
 '거',
 '를',
 ' 분석',
 '하는',
 ' 금',
 '융',
 '/',
 '경제',
 ' 분석',
 ' 전문',
 '가',
 '입니다',
 '.\n\n',
 '다',
 '음',
 ' 출력',
 '지',
 '시',
 '사항',
 '을',
 ' 지',
 '켜',
 '주세요',
 '.\n',
 '1',
 '.',
 ' 뉴',
 '스',
 '와',
 ' 종',
 '목',
 '간',
 '의',
 ' 연',
 '관',
 '성을',
 ' 발견',
 '할',
 ' 수',
 ' 없',
 '다면',
 ':\n',
 '   ',
 ' -',
 ' stock',
 '_related',
 '를',
 ' False',
 '로',
 ' 작성',
 '하세요',
 '.\n',
 '   ',
 ' -',
 ' summary',
 '에',
 ' 뉴',
 '스의',
 ' 요',
 '약',
 '을',
 ' 작성',
 '하세요',
 '.\n',
 '2',
 '.',
 ' 뉴',
 '스',
 '와',
 ' 종',
 '목',
 '간',
 '의',
 ' 연',
 '관',
 '성을',
 ' 발견',
 '했다',
 '면',
 ':\n',
 '   ',
 ' -',
 ' stock',
 '_related',
 '를',
 ' True',
 '로',
 ' 작성',
 '하세

In [18]:
# 데이터프레임 시각화

import pandas as pd

df = pd.DataFrame({
    'token':text_tokens, # 토큰 (디코딩된 문자열)
    'input_ids' : batch['input_ids'][0].tolist(), # 입력 토큰 ID
    'attention_mask' : batch['attention_mask'][0].tolist(), # 패딩 여부(1/0)
    'labels':batch['labels'][0].tolist() # 정답 라벨 (마스킹: -100)
}).transpose()

pd.set_option('display.max_columns', None) # 컬럼 생략 없음
df


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299,300,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,326,327,328,329,330,331,332,333,334,335,336,337,338,339,340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383,384,385,386,387,388,389,390,391,392,393,394,395,396,397,398,399,400,401,402,403,404,405,406,407,408,409,410,411,412,413,414,415,416,417,418,419,420,421,422,423,424,425,426,427,428,429,430,431,432,433,434,435,436,437,438,439,440,441,442,443,444,445,446,447,448,449,450,451,452,453,454,455,456,457,458,459,460,461,462,463,464,465,466,467,468,469,470,471,472,473,474,475,476,477,478,479,480,481,482,483,484,485,486,487,488,489,490,491,492,493,494,495,496,497,498,499,500,501,502,503,504,505,506,507,508,509,510,511,512,513,514,515,516,517,518,519,520,521,522,523,524,525,526,527,528,529,530,531,532,533,534,535,536,537,538,539,540,541,542,543,544,545,546,547,548,549,550,551,552,553,554,555,556,557,558,559,560,561,562,563,564,565,566,567,568,569,570,571,572,573,574,575,576,577,578,579,580,581,582,583,584,585,586,587,588,589,590,591,592,593,594,595,596,597,598,599,600,601,602,603,604,605,606,607,608,609,610,611,612,613,614,615,616,617,618,619,620,621,622,623,624,625,626,627,628,629,630,631,632,633,634,635,636,637,638,639,640,641,642,643,644,645,646,647,648,649,650,651,652,653,654,655,656,657,658,659,660,661,662,663,664,665,666,667,668,669,670,671,672,673,674,675,676,677,678,679,680,681,682,683,684,685,686,687,688,689,690,691,692,693,694,695,696,697,698,699,700,701,702,703,704,705,706,707,708,709,710,711,712,713,714,715,716,717,718,719,720,721,722,723,724,725,726,727,728,729,730,731,732,733,734,735,736,737,738,739,740,741,742,743,744,745,746,747,748,749,750,751,752,753,754,755,756,757,758,759,760,761,762,763,764,765,766,767,768,769,770,771,772,773,774,775,776,777,778,779,780,781,782,783,784,785,786,787,788,789,790,791,792,793,794,795,796,797,798,799,800,801,802,803,804,805,806,807,808,809,810,811,812,813,814,815,816,817,818,819,820,821,822,823,824,825,826,827,828,829,830,831,832,833,834,835,836,837,838,839,840,841,842,843,844,845,846,847,848,849,850,851,852,853,854,855,856,857,858,859,860,861,862,863,864,865,866,867,868,869,870,871,872,873,874,875,876,877,878,879,880,881,882,883,884,885,886,887,888,889,890,891,892,893,894,895,896,897,898,899,900,901,902,903,904,905,906,907,908,909,910,911,912,913,914,915,916,917,918,919,920,921,922,923,924,925,926,927,928,929,930,931,932,933,934,935,936,937,938,939,940,941,942,943,944,945,946,947,948,949,950,951,952,953,954,955,956,957,958,959,960,961,962,963,964,965,966,967,968,969,970,971,972,973,974,975,976,977,978,979,980,981,982,983,984,985,986,987,988,989,990,991,992,993,994,995,996,997,998,999,1000,1001,1002,1003,1004,1005,1006,1007,1008,1009,1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,1021

## PEFT Finetuning - LoRA

* LoRA는 **"Low-Rank Adapter(저랭크 어댑터)"**
* 거대한 대형언어모델(LLM)의 **전체 파라미터를 일일이 미세조정(파인튜닝)하지 않고**,
  **딱 필요한 핵심 부분만 저렴하게 빠르게 학습**하는 최신 파인튜닝.
* **"LLM의 성능은 그대로, 비용/시간/메모리/유지보수는 최소로"** 파인튜닝을 할 수 있게 해주는 AI 실무에서 가장 중요한 기법 중 하나이다.

**왜 LoRA가 등장했을까?**

* GPT, Llama, DeepSeek 같은 대형언어모델은 **파라미터(매개변수) 수가 수십억\~수조 개**나 된다.
* 이런 모델을 파인튜닝하려면 **막대한 GPU 메모리와 시간, 저장 공간**이 필요.
* 하지만, 실제로 특정 태스크에 맞게 모델을 조정할 때 **전체를 다 바꿀 필요가 없다.**
* 대부분의 정보는 기존 모델에 이미 들어있고,
  **특정 입력(질문)과 특정 출력(답변)의 관계만 살짝 조정**해주면 충분하다.

**LoRA의 원리**

* 기존 대형 모델의 핵심 연산(주로 "곱셈" 부분)에
  **작고 얇은 "보조 네트워크(어댑터 레이어)"**를 덧붙인다.
* 전체 모델은 거의 건드리지 않고,
  **이 어댑터 레이어의 파라미터만 새로 추가해서 학습**
* 학습이 끝나면,

  * 원본 모델은 그대로
  * 어댑터(작은 추가 파라미터)만 별도로 저장하면 끝!
* 추론할 땐 **원본 모델 + LoRA 어댑터**를 합쳐서 쓸 수 있다.

**LoRA의 장점**

* **파인튜닝 비용(시간, 메모리, 저장 용량)이 압도적으로 절약**된다.
* 7B, 13B, 70B 등 대형 모델도
  **일반 GPU(24GB/48GB)로도 쉽게 파인튜닝**이 가능하다.
* **동일한 원본 모델에 다양한 LoRA 어댑터만 바꿔 끼우며
  다양한 분야별 파인튜닝 결과를 쉽게 쓸 수 있다.**

**LoRA와 기존 방식의 비교**

* **기존 파인튜닝:**
  전체 파라미터(수십\~수백 GB)를 새로 저장/관리/학습 → 비효율적
* **LoRA:**
  원본은 그대로 두고,
  변화가 필요한 부분(수 MB\~수십 MB)만 별도로 학습/저장


**실전에서의 활용 예시**

* 번역 LoRA, 요약 LoRA, 감정분석 LoRA 등
  **하나의 원본 모델에 여러 용도별 어댑터를 저장/관리**할 수 있다.
* **A100 80GB, 3090, T4 등 다양한 GPU 환경에서도
  고성능 LLM 튜닝이 매우 쉽게 가능하다.**

In [19]:
# LoRA 설정 적용 후 학습 가능한 파라미터(Trainable) 확인
from peft import LoraConfig, get_peft_model # LoRA 설정 / 적용 함수

lora_config = LoraConfig(
    r = 8,              # 저랭크 행렬 rank
    lora_alpha=32,      # LoRA 스케일 계수 (alpha/r)
    lora_dropout=0.1,
    bias = "none",      # bias 학습 제외
    target_modules=['q_proj', 'v_proj'], # LoRA 를 주입할 모듈 (Q/V Projection)
    task_type = "CAUSAL_LM" # 작업 유형: 생성형
)

model = get_peft_model(model, lora_config) # 기존 모델에 LoRA 어댑터 적용
model.print_trainable_parameters() # 학습 가능한 파라미터 수 (비율) 

trainable params: 3,407,872 || all params: 8,033,669,120 || trainable%: 0.0424


In [20]:
from trl import SFTConfig  # TRL SFT 학습 설정 클래스

hub_model_id = 'kkkfu/Llama-VARCO-8b-news2stock-analyzer'  # 학습 완료 후 업로드할 Hub 모델 ID

sft_config = SFTConfig(  # SFT 학습 하이퍼파라미터/저장/로그 설정
    output_dir="Llama-VARCO-8b-news2stock-analyzer", # 학습 완료된 모델과 체크포인트가 저장될 경로이다.
    num_train_epochs=3,                              # 전체 데이터셋을 반복 학습할 횟수(Epoch)이다.
    per_device_train_batch_size=2,                   # 각 GPU(장치)당 한 번에 처리할 데이터 샘플의 개수이다.
    gradient_accumulation_steps=2,                   # 그래디언트를 2번 누적한 후 가중치를 업데이트한다. (실제 배치 크기 = 2 * 2 = 4 효과를 낸다.)
    gradient_checkpointing=True,                     # VRAM 절약을 위해 중간 활성화 값을 저장하지 않고 역전파 시 재계산하는 설정이다.
    optim="adamw_torch_fused",                       # 최적화 알고리즘 설정이다. fused 버전은 CUDA에서 더 빠르다.
    logging_steps=10,                                # 10 스텝마다 학습 로그(Loss 등)를 출력한다.
    save_strategy="steps",                           # 체크포인트 저장 기준을 'steps'(스텝 수)로 설정한다. (옵션: 'epoch')
    save_steps=50,                                   # 50 스텝마다 모델 체크포인트를 저장한다.
    bf16=True,                                       # BF16(Brain Float 16) 정밀도를 사용하여 메모리를 아끼고 연산 속도를 높인다. (Ampere GPU 이상 권장)
    learning_rate=1e-4,                              # 학습률(Learning Rate)이다. 가중치 업데이트의 크기를 결정한다.
    max_grad_norm=0.3,                               # 그래디언트 클리핑 임계값이다. 그래디언트 폭주를 막아 학습 안정성을 높인다.
    warmup_steps=0.03,                               # 전체 학습 단계의 3% 동안 학습률을 서서히 올리는 웜업(Warmup)을 수행한다.
    lr_scheduler_type="constant_with_warmup",        # 학습률 스케줄러 타입이다. 여기서는 학습률을 변동 없이 상수로 유지한다.
    push_to_hub=True,                                # 학습이 끝나면 Hugging Face Hub에 모델을 자동으로 업로드한다.
    hub_model_id=hub_model_id,                       # Hub에 업로드될 때 사용될 저장소(Repository) ID이다.
    hub_token=True,                                  # Hub 업로드를 위해 인증 토큰을 사용한다.
    remove_unused_columns=False,                     # 데이터셋에서 모델의 forward 메서드 시그니처에 없는 컬럼을 자동으로 삭제하지 않도록 한다.
    dataset_kwargs={"skip_prepare_dataset": True},   # 데이터셋 처리 과정(packing 등)을 건너뛰도록 하는 설정이다.
    report_to=['wandb'],                               # 학습 기록을 전송할 툴(WandB, Tensorboard 등)을 지정한다. 빈 리스트는 기록하지 않음을 의미한다.
    label_names=["labels"],                          # 손실(Loss) 계산 시 정답(Target)으로 사용할 데이터셋의 컬럼 이름이다.
    loss_type="nll"                                  # chunked_nll 비활성화 한다
)

In [21]:
from trl import SFTTrainer # SFT 학습용 Trainer

trainer = SFTTrainer(
    model = model,                   # LoRA 적용된 학습 모델
    args = sft_config,               # SFT 학습 설정
    train_dataset = train_dataset,   # 학습 데이터 셋
    data_collator = data_collator    # 배치 텐서 생성 함수 
)

trainer.train() # 학습 실행

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 0}.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice:  1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.


wandb: Paste your API key and hit enter:  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: free-rjs103 (free-rjs103-palydata) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


torch.Size([1, 2112])
[128006, 78191, 128007, 198]
[128009]
0: 1614 ~ 2112
torch.Size([1, 1898])
[128006, 78191, 128007, 198]
[128009]
0: 1106 ~ 1898
torch.Size([1, 1017])
[128006, 78191, 128007, 198]
[128009]
0: 562 ~ 1017


Step,Training Loss
10,1.849098
20,1.721304
30,1.488012
40,1.292434
50,1.277813
60,1.280565
70,1.197452
80,1.177836
90,1.060247
100,1.112291


torch.Size([1, 1075])
[128006, 78191, 128007, 198]
[128009]
0: 738 ~ 1075
torch.Size([1, 853])
[128006, 78191, 128007, 198]
[128009]
0: 633 ~ 853
torch.Size([1, 2243])
[128006, 78191, 128007, 198]
[128009]
0: 1612 ~ 2243
torch.Size([1, 1576])
[128006, 78191, 128007, 198]
[128009]
0: 1288 ~ 1576
torch.Size([1, 2121])
[128006, 78191, 128007, 198]
[128009]
0: 1398 ~ 2121
torch.Size([1, 649])
[128006, 78191, 128007, 198]
[128009]
0: 328 ~ 649
torch.Size([1, 645])
[128006, 78191, 128007, 198]
[128009]
0: 333 ~ 645
torch.Size([1, 1519])
[128006, 78191, 128007, 198]
[128009]
0: 963 ~ 1519
torch.Size([1, 1281])
[128006, 78191, 128007, 198]
[128009]
0: 862 ~ 1281
torch.Size([1, 1318])
[128006, 78191, 128007, 198]
[128009]
0: 825 ~ 1318
torch.Size([1, 4821])
[128006, 78191, 128007, 198]
[128009]
0: 4047 ~ 4821
torch.Size([1, 1786])
[128006, 78191, 128007, 198]
[128009]
0: 1224 ~ 1786
torch.Size([1, 1194])
[128006, 78191, 128007, 198]
[128009]
0: 967 ~ 1194
torch.Size([1, 1065])
[128006, 78191, 1

TrainOutput(global_step=600, training_loss=1.0903061135609944, metrics={'train_runtime': 473.2478, 'train_samples_per_second': 5.071, 'train_steps_per_second': 1.268, 'total_flos': 7.432055393685504e+16, 'train_loss': 1.0903061135609944, 'epoch': 3.0})

# 평가

In [22]:
# 테스트셋 messages에서 프롬프트/정답(assistant) 텍스트 분리
prompt_list = [] # 프롬프트 (assistant 답변 내용 이전) 리스트
label_list = [] # 정답 (assistant 답변 내용) 리스트

for messages in test_dataset["messages"]:
    # 채팅 템플릿 문자열로 반환
    text = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt =False)
    # assistant 답변 내용 전까지 input
    input = text.split('<|start_header_id|>user<|end_header_id|>\n')[0] + \
        '<|start_header_id|>user<|end_header_id|>\n'
    # assistant 답변 내용(종료 토큰 전) 추출
    labels = text.split('<|start_header_id|>user<|end_header_id|>\n')[1].split('<|eot.id|>')[0]
    prompt_list.append(input)
    label_list.append(labels)

In [23]:
print(prompt_list[100])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,
특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.

다음 출력지시사항을 지켜주세요.
1. 뉴스와 종목간의 연관성을 발견할 수 없다면:
    - stock_related를 False로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
2. 뉴스와 종목간의 연관성을 발견했다면:
    - stock_related를 True로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.
    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.
    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.<|eot_id|><|start_header_id|>user<|end_header_id|>



In [24]:
print(label_list[100])


공유수면 사용하려면 어입인 의견 들어야
해수부 5일 개정 공유수면 관리 및 매립에 관한 법률 시행 헤럴드경제 홍태화 기자 앞으로 공유수면관리청이 어업·환경 등에 영향을 미칠 것으로 예상되는 공유수면 점용·사용 허가를 할 때 미리 어업인 등 이해관계자들의 의견을 들어야 한다. 해양수산부는 5일 이같은 내용이 담긴 개정 공유수면 관리 및 매립에 관한 법률과 같은 법 시행령·시행규칙이 이날부터 시행된다고 밝혔다. 바다·바닷가·하천 등 공유수면은 공유재이기 때문에 이를 점용·사용하기 위해서는 별도의 허가를 받아야 한다. 최근 해상풍력 발전시설 해변을 이용한 관광시설 등 대규모 시설이 공유수면을 장기적으로 점용·사용하는 경우가 늘어났지만 이해 관계자의 의견을 사전에 수렴할 수 없는 문제가 있었다. 이에 공유수면 점용·사용으로 인한 사회적 갈등이 증가했다. 이러한 문제를 해결하기 위해 해수부는 지난 1월 공유수면 점용·사용 허가를 할 때 이해관계자의 의견을 듣도록 공유수면 관리 및 매립에 관한 법률을 개정했다. 법 개정에 따라 공유수면관리청이 해양환경·수산자원·자연경관 보호 등에 영향을 끼칠 수 있는 공유수면 점용·사용 신청을 받은 경우 이를 관보 공보 와 인터넷 홈페이지에 공고해야 한다. 또 점용·사용 허가를 했을 때 피해를 볼 것으로 예상되는 어업인에 대한 의견 조사도 별도로 진행해야 한다. 황준성 해수부 해양공간정책과장은 공유수면 점용·사용으로 인한 이해 관계자의 피해를 방지하려는 법령 개정의 취지를 달성할 수 있도록 각 공유수면관리청과 협력해 관련 제도의 차질 없는 운영을 지원하겠다 고 말했다.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{"stock_related":true,"summary":"해양수산부가 공유수면 관리 및 매립에 관한 법률과 시행령·시행규칙 개정안을 시행했다. 이에 따라 해상풍력 발전시설, 해변 관광시설 등 해양환경·수산자원·자연경관에 영향을 줄 수 있는 공유수면 점용·사용 허가를 진행

### 추론모델 - 런타임결합
1. lora모델을 로드
2. base모델 + lora adapter 세팅

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer, pipeline # 토크나이저 자동 로더 / 파이프라인
import torch

peft_model_name = hub_model_id # 업로드된 PEFT 모델 repo

finetuned_model = AutoPeftModelForCausalLM.from_pretrained(
    peft_model_name, 
    dtype = torch.bfloat16, # 가중치 로딩 dtype(bf16)
    device_map = 'auto', # 환경에 맞춰 CPU / GPU 자동 배치
    quantization_config = quant_config # 4bit 양자화 설정 적용
)
# 해당 모델의 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(peft_model_name)

# 텍스트 생성 파이프라인
pipe = pipeline('text-generation', model=finetuned_model, tokenizer=tokenizer)
pipe

adapter_config.json:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/497 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/348 [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 6.83MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

TextGenerationPipeline: {'model': 'PeftModelForCausalLM', 'dtype': 'bfloat16', 'device': 'cuda', 'input_modalities': 'text', 'output_modalities': ('text',)}

In [27]:
# <|eot_id|>의 토큰 id 추출
eos_token = tokenizer('<|eot_id|>', add_special_tokens=False)['input_ids'][0]
eos_token

128009

In [28]:
# 테스트 추론 함수 : 프롬프트, 정답, 모델응답 비교 출력
def test_inference(pipe, prompt):
    # 파이프라인으로 결정론적인 답변 생성
    outputs = pipe(prompt, max_new_tokens=1024, eos_token_id=eos_token, do_sample=False)
    assistant_start = len(prompt) # 입력 구간 이후 생성 결과만 사용
    # 프롬프트 이후(생성된 부분)만 반환
    return outputs[0]['generated_text'][assistant_start:].strip() 

for prompt, label in zip(prompt_list[10:13], label_list[10:13]): # 10~12 샘플
    print(f"[prompt]: {prompt}")
    print(f"[label] : {label}")
    print(f"[response]: {test_inference(pipe,prompt)}")
    print('='*100)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[prompt]: <|begin_of_text|><|start_header_id|>system<|end_header_id|>

당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,
특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.

다음 출력지시사항을 지켜주세요.
1. 뉴스와 종목간의 연관성을 발견할 수 없다면:
    - stock_related를 False로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
2. 뉴스와 종목간의 연관성을 발견했다면:
    - stock_related를 True로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.
    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.
    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.<|eot_id|><|start_header_id|>user<|end_header_id|>

[label] : 
글로벌 비즈 가트너 올해 전세계 스마트폰 판매량 7% 감소 전망
경제와이드 모닝벨 글로벌 비즈 임선우 외신캐스터 글로벌 비즈입니다. ◇ 올해 스마트폰 판매 감소 올해 전세계 스마트폰 판매량이 크게 줄어들 것이란 전망이 나왔습니다. 시장조사업체 가트너는 글로벌 스마트폰 판매가 7% 하락할 것으로 내다봤는데요. 경제 전반에 걸친 침체 우려와 중국의 봉쇄조치 여파 그리고 인플레이션으로 소비자들이 지갑을 열기 주저하면서 수요가 줄어들 것 이라고 설명했습니다. 그러면서 올해 전체 출하량은 14억6천만대 수준에 그칠 것으로 예측했는데요. 종전 전망치인 16억대에서 대폭 낮춰 잡았습니다. 특히 세계 최대 스마트폰 시장인 중국에서 판매량은 18%가 감소할 

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[response]: {"news_title":"미국 경제 성장률 2.7%로 2023년 1분기 GDP 성장률 전망 상승","news_date":"2023년 4월 26일","news_content":"미국 경제성장률이 2023년 1분기 2.7%로 전망됐다. 이는 2022년 4분기 2.6%였던 전망치를 0.1% 높인 수치다. 미국 연방준비제도(Fed)는 인플레이션 둔화와 실업률 하락을 기반으로 2023년 1분기 기준 금리 인하 전망을 발표했다. 미국 경제가 2023년 1분기에도 강한 성장세를 유지할 경우, 미국 소비자와 기업의 실질 소득 증가가 이어질 수 있어 소비와 투자 확대에 긍정적이다. 다만 2.7% 성장률은 2023년 전망치인 2.8%보다 낮아 금리 인하의 효과가 제한적일 수 있고, 인플레이션 둔화가 지속되지 않으면 금리 인하가 중단될 가능성도 있다.","stock_related":true,"positive_stocks":["JPMorgan Chase","Bank of America","Citi","Wells Fargo","Visa","Mastercard","American Express","Visa","Mastercard","American Express"],"positive_keywords":["미국 경제 성장률 상승","소비 확대","금리 인하","실질 소득 증가","소비 지출","신용 활성화","카드 거래","금융 서비스"],"positive_reasons":"미국 경제가 2023년 1분기에도 2.7% 성장할 경우 소비자와 기업의 실질 소득이 증가하고, 금리 인하가 이어질 경우 대출 및 신용 활성화가 확대될 수 있다. 이는 은행의 대출 수익, 신용카드 거래, 소비자 신용 및 투자 활성화에 긍정적이다. 특히 금리 인하가 지속될 경우 대출금리 하락과 함께 대출 수익성 개선이 기대되며, 신용카드 회사는 소비 증가와 신용 활성화로 카드 거래 및 결제 수수료 수익이 늘어날 가능성이 있다. 다만 금리 인하가 실제로 실현되고 대출 및 신용 활성화가 확대되는지는 

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[response]: {"news_title":"미국 경제 성장률 2.7%로 2023년 1분기 GDP 성장률 전망 상승","news_date":"2023년 4월 26일","news_content":"미국 경제성장률이 2023년 1분기 2.7%로 전망됐다. 이는 2022년 4분기 2.6%였던 전망치를 0.1% 높인 수치다. 미국 연방준비제도(Fed)는 인플레이션 둔화와 실업률 하락을 기반으로 2023년 1분기 기준 금리 인하 전망을 발표했다. 미국 경제가 2023년 1분기에도 강한 성장세를 유지할 경우, 미국 소비자와 기업의 실질 소득 증가가 이어질 수 있어 소비와 투자 확대에 긍정적이다. 다만 2.7% 성장률은 2023년 전망치인 2.8%보다 낮아 금리 인하의 효과가 제한적일 수 있고, 인플레이션 둔화가 지속되지 않으면 금리 인하가 중단될 가능성도 있다.","stock_related":true,"positive_stocks":["JPMorgan Chase","Bank of America","Citi","Wells Fargo","Visa","Mastercard","American Express","Visa","Mastercard","American Express"],"positive_keywords":["미국 경제 성장률 상승","소비 확대","금리 인하","실질 소득 증가","소비 지출","신용 활성화","카드 거래","금융 서비스"],"positive_reasons":"미국 경제가 2023년 1분기에도 2.7% 성장할 경우 소비자와 기업의 실질 소득이 증가하고, 금리 인하가 이어질 경우 대출 및 신용 활성화가 확대될 수 있다. 이는 은행의 대출 수익, 신용카드 거래, 소비자 신용 및 투자 활성화에 긍정적이다. 특히 금리 인하가 지속될 경우 대출금리 하락과 함께 대출 수익성 개선이 기대되며, 신용카드 회사는 소비 증가와 신용 활성화로 카드 거래 및 결제 수수료 수익이 늘어날 가능성이 있다. 다만 금리 인하가 실제로 실현되고 대출 및 신용 활성화가 확대되는지는 

### 추론모델 사전 병합
- 진행 후에는 PEFT 없이 해당 repo 모델로 바로 로드/추론 가능하다

In [29]:
from peft import AutoPeftModelForCausalLM # PEFT (LoRA) 모델 로더
from transformers import AutoTokenizer, pipeline

merged_model_id = 'kkkfu/Llama-VARCO-8b-news2stock-analyzer-4bit-merged'

tokenizer = AutoTokenizer.from_pretrained(peft_model_name)

# LoRA 어댑터를 포함한 모델 로드
finetuned_model = AutoPeftModelForCausalLM.from_pretrained(
    peft_model_name,
    dtype = torch.bfloat16, # 가중치 로딩 dtype(bf16)
    device_map= 'auto',      # 환경에 맞춰 CPU / GPU 자동 배치
    quantization_config = quant_config # 4bit 양자화 설정 적용
)

merged_model = finetuned_model.merge_and_unload() # LoRA 가중치를 베이스 모델에 병합 후 어댑터 언로드

merged_model.push_to_hub(merged_model_id, token=True) # 병합 모델 Hub 업로드
tokenizer.push_to_hub(merged_model_id, token =True) # 토크나이저 Hub 업로드

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:377: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/kkkfu/Llama-VARCO-8b-news2stock-analyzer-4bit-merged/commit/b194a8deee21a328b7eae492812af7a0158b9892', commit_message='Upload tokenizer', commit_description='', oid='b194a8deee21a328b7eae492812af7a0158b9892', pr_url=None, repo_url=RepoUrl('https://huggingface.co/kkkfu/Llama-VARCO-8b-news2stock-analyzer-4bit-merged', endpoint='https://huggingface.co', repo_type='model', repo_id='kkkfu/Llama-VARCO-8b-news2stock-analyzer-4bit-merged'), pr_revision=None, pr_num=None)

In [30]:
#GPU 메모리 해제 (객체 삭제 + GC + CUDA 캐시 비우기) - 필요시 주석 해제 후 사용
del finetuned_model, pipe, merged_model # 사용 완료된 참조 제거

import gc       # 가비지 컬렉션 모듈
gc.collect()    # 참조가 끊긴 객체 메모리 정리

torch.cuda.empty_cache() # CUDA 캐시 메모리 비움

In [31]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id = 'kkkfu/Llama-VARCO-8b-news2stock-analyzer-4bit-merged'

# 병합된 단일 모델 로드 (PEFT 불필요)
finetuned_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype = torch.bfloat16,
    device_map = 'auto' 
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
pipe = pipeline('text-generation', model = finetuned_model, tokenizer=tokenizer)
pipe


config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 5.70GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/143 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/497 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/348 [00:00<?, ?B/s]

TextGenerationPipeline: {'model': 'LlamaForCausalLM', 'dtype': 'bfloat16', 'device': 'cuda', 'input_modalities': 'text', 'output_modalities': ('text',)}

In [32]:
for prompt, label in zip(prompt_list[10:13], label_list[10:13]): # 10~12 샘플
    print(f"[prompt]: {prompt}")
    print(f"[label] : {label}")
    print(f"[response]: {test_inference(pipe,prompt)}")
    print('='*100)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[prompt]: <|begin_of_text|><|start_header_id|>system<|end_header_id|>

당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,
특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.

다음 출력지시사항을 지켜주세요.
1. 뉴스와 종목간의 연관성을 발견할 수 없다면:
    - stock_related를 False로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
2. 뉴스와 종목간의 연관성을 발견했다면:
    - stock_related를 True로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.
    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.
    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.<|eot_id|><|start_header_id|>user<|end_header_id|>

[label] : 
글로벌 비즈 가트너 올해 전세계 스마트폰 판매량 7% 감소 전망
경제와이드 모닝벨 글로벌 비즈 임선우 외신캐스터 글로벌 비즈입니다. ◇ 올해 스마트폰 판매 감소 올해 전세계 스마트폰 판매량이 크게 줄어들 것이란 전망이 나왔습니다. 시장조사업체 가트너는 글로벌 스마트폰 판매가 7% 하락할 것으로 내다봤는데요. 경제 전반에 걸친 침체 우려와 중국의 봉쇄조치 여파 그리고 인플레이션으로 소비자들이 지갑을 열기 주저하면서 수요가 줄어들 것 이라고 설명했습니다. 그러면서 올해 전체 출하량은 14억6천만대 수준에 그칠 것으로 예측했는데요. 종전 전망치인 16억대에서 대폭 낮춰 잡았습니다. 특히 세계 최대 스마트폰 시장인 중국에서 판매량은 18%가 감소할 

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[response]: Summary:

글로벌 경제는 2023년에도 지속적인 침체로 접어들 것으로 전망되며, 미국의 연방준비제도(Fed)는 인플레이션을 억제하기 위해 추가적인 금리 인상 전망이다. 이는 세계 각국의 경제성장 둔화와 함께 국제통화기금(IMF)의 세계 경제 성장률 전망을 낮추는 요인이다. 

Stock-related:

stock_related: True

Positive Stocks:

positive_stocks: 
- 
  stock: 
    - "AAPL" (Apple Inc.)
    - "MSFT" (Microsoft Corporation)
    - "AMZN" (Amazon.com Inc.)
    - "GOOGL" (Alphabet Inc.)
    - "V" (Visa Inc.)
  positive_keywords: 
    - "디지털화"
    - "클라우드 컴퓨팅"
    - "온라인 쇼핑"
    - "데이터 분석"
    - "디지털 결제"
  positive_reasons:
    - "글로벌 경제 침체에도 불구하고, 위의 기업들은 디지털화와 클라우드 컴퓨팅, 온라인 쇼핑 등에 대한 수요가 상대적으로 높아질 가능성이 있다. 특히, 'AAPL'의 Apple은 개인용 컴퓨터와 관련 제품의 수요가 상대적으로 안정적일 수 있으며, 'MSFT'의 Microsoft는 기업 클라우드 솔루션에 대한 수요가 증가할 수 있다. 'AMZN'의 Amazon은 온라인 쇼핑에 대한 수요가 지속될 것이고, 'GOOGL'의 Alphabet(구글)은 데이터 분석 및 디지털 마케팅 서비스에 대한 수요가 높아질 수 있다. 'V'의 Visa는 디지털 결제의 확산으로 인해 수요가 증가할 수 있다."

Negative Stocks:

negative_stocks:
- 
  stock: 
    - "NIO" (NIO Inc.)
    - "LYFT" (Lyft, Inc.)
    - "ZM" (Zoom Video Communications, Inc.)
  

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[response]: Summary:

글로벌 경제는 2023년에도 지속적인 침체로 접어들 것으로 전망되며, 미국의 연방준비제도(Fed)는 인플레이션을 억제하기 위해 추가적인 금리 인상 전망이다. 이는 세계 각국의 경제성장 둔화와 함께 국제통화기금(IMF)의 세계 경제 성장률 전망을 낮추는 요인이다. 

Stock-related:

stock_related: True

Positive Stocks:

positive_stocks: 
- 
  stock: 
    - "AAPL" (Apple Inc.)
    - "MSFT" (Microsoft Corporation)
    - "AMZN" (Amazon.com Inc.)
    - "GOOGL" (Alphabet Inc.)
    - "V" (Visa Inc.)
  positive_keywords: 
    - "디지털화"
    - "클라우드 컴퓨팅"
    - "온라인 쇼핑"
    - "데이터 분석"
    - "디지털 결제"
  positive_reasons:
    - "글로벌 경제 침체에도 불구하고, 위의 기업들은 디지털화와 클라우드 컴퓨팅, 온라인 쇼핑 등에 대한 수요가 상대적으로 높아질 가능성이 있다. 특히, 'AAPL'의 Apple은 개인용 컴퓨터와 관련 제품의 수요가 상대적으로 안정적일 수 있으며, 'MSFT'의 Microsoft는 기업 클라우드 솔루션에 대한 수요가 증가할 수 있다. 'AMZN'의 Amazon은 온라인 쇼핑에 대한 수요가 지속될 것이고, 'GOOGL'의 Alphabet(구글)은 데이터 분석 및 디지털 마케팅 서비스에 대한 수요가 높아질 수 있다. 'V'의 Visa는 디지털 결제의 확산으로 인해 수요가 증가할 수 있다."

Negative Stocks:

negative_stocks:
- 
  stock: 
    - "NIO" (NIO Inc.)
    - "LYFT" (Lyft, Inc.)
    - "ZM" (Zoom Video Communications, Inc.)
  

# 추론

In [33]:
# 뉴스 1건 입력받아 추론하는 함수

def inference(news) :
    messages = [
        {'role': 'system', 'content': '''
당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,
특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.

다음 출력지시사항을 지켜주세요.
1. 뉴스와 종목간의 연관성을 발견할 수 없다면:
    - stock_related를 False로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
2. 뉴스와 종목간의 연관성을 발견했다면:
    - stock_related를 True로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.
    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.
    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.
'''},  # 시스템 지시문
        {'role': 'user', 'content': news}
    ]
    
    # messages를 채팅 프롬프트 문자열로 반환
    prompt = tokenizer.apply_chat_template(messages, tokenize=False)
    outputs = pipe(prompt, max_new_tokens=1024, eos_token_id=eos_token, do_sample=False)
    assistant_start = len(prompt) # 입력 구간 이후 생성 결과만 사용
    # 프롬프트 이후(생성된 부분)만 반환
    return outputs[0]['generated_text'][assistant_start:].strip() 

In [34]:
news = ''' 
엔화, 반년 만에 최강세…"추가 상승 가능성 크다"

달러당 153엔대까지 상승
美日 추가 개입 가능성
금리인상 전망 등 영향
일본 엔화 가치가 달러당 153엔대까지 가파르게 상승했다. 지난 2월 중순 이후 반년 만에 가장 높은 수준이다. 일본 정부와 일본은행(BOJ)의 외환시장 개입 당시에도 넘지 못했던 달러당 155엔 선을 돌파한 이후에도 오름세가 이어지고 있다. 올해 고점인 152엔대까지 상승할 여지가 있다는 관측이 나온다.

8일 니혼게이자이신문(닛케이)에 따르면 이날 오전 도쿄 외환시장에서 엔화 가치는 장중 달러당 153엔대까지 뛰었다. 전날 달러당 155엔 선을 넘어섰으나, 이후에도 매수세가 지속됐다. 달러당 153엔대는 지난 2월 중순 이후 최고 수준이다. 지난 4~5월과 7월, 일본 당국이 외환시장 개입에 나섰을 당시에도 이 수준은 넘지 못했다.

이번 엔화 강세에는 미국과 일본 통화당국이 추가로 외환시장에 개입할 수 있다는 가능성과, BOJ의 금리 인상 가속화 전망 등이 복합적으로 영향을 미친 것으로 보인다고 닛케이는 짚었다.

먼저 외환시장에서는 이달 이후 미·일 통화당국이 엔화 약세 시정을 위한 구체적인 조치를 취할 것이라는 전망이 나오고 있다. 스콧 베선트 미국 재무부 장관은 지난달 30일 우에다 가즈오 BOJ 총재와 만나 "엔화의 대폭적인 저평가에 대처하기 위해 일본이 단호한 시장·금융 정책상 조치를 취하는 것을 강력히 지지한다"고 발언한 바 있다. 가타야마 사쓰키 일본 재무상도 과도한 엔화 약세가 이어질 경우 미국과 협조해 추가 개입에 나설 것을 시사해왔다.

BOJ의 금리 인상 속도가 빨라질 것이라는 전망도 엔화 매수세를 부추기고 있다. 시장에서는 BOJ가 오는 17~18일 열리는 금융정책결정회의에서 기준금리를 0.25%포인트 인상할 가능성을 선반영하고 있다. 여기에 시장은 추가 금리 상승 가능성에 베팅하고 있다. 닛케이는 "이번 기준금리 인상 이후에도 3개월에 한 번 정도의 속도로 금리 인상을 이어가거나, 최종 기준금리 목표치가 상향 조정될 것이라는 전망이 확산하고 있다"고 전했다.

여기에 중동 정세 긴장이 완화될 것이라는 기대감도 엔화 강세에 힘을 보태고 있다. 에스마일 바가이 이란 외무부 대변인은 전날 호르무즈 해협의 임시 항로를 둘러싼 오만과의 협상이 최종 단계에 도달했으며, 빠르면 며칠 내로 합의에 이를 전망이라고 밝힌 바 있다. 이로 인해 안전자산 선호 현상으로 나타났던 '유사시 달러 매수 현상'도 주춤해진 분위기다.

엔화 강세가 나타나면서 엔저에 베팅하던 투자자들의 포지션 청산도 가속화되고 있다. 엔화 강세로 손실이 커지자, 달러를 팔고 엔화를 되사들이면서 상승세를 더 부추기는 모습이다. 미쓰비시UFJ신탁은행 자금·외환부의 오카다 유스케 상급조사역은 "헤지펀드(같은 단기 투기 세력)뿐만 아니라 중장기적 관점을 가지고 거래하는 주체들도 엔 매도·달러 매수 포지션을 청산하는 등, 최근 추세가 변화하고 있다"고 닛케이에 전했다.

여기에 달러당 155엔 선이 무너지면서 손절매까지 잇따랐다. 블룸버그통신은 익명의 트레이더를 인용해 155엔 아래에 설정돼 있던 대규모 손절매 주문이 엔화가 상승하며 실행됐고, 옵션 딜러들도 달러 매도에 나서면서 상승세에 힘을 실었다고 분석했다.

이렇게 복합적인 요인들이 겹치면서 이번 엔화 급상승은 지난번 당국의 직접적인 외환 시장 개입에 따른 상승과는 성격이 다르다는 평가도 나온다. 반 루 러셀인베스트먼츠 글로벌 채권·외환 솔루션 전략 책임자는 "첫 번째 시장 개입의 효과는 이미 사라진 것으로 보인다. 이번 두 번째 상승은 시장 자체의 힘으로 나타나는 것으로 보고, 그렇기에 이번 움직임이 훨씬 더 중요하다"고 블룸버그에 전했다.

추가 상승 여력이 있다는 전망도 나왔다. 우에노 다이사쿠 미쓰비시UFJ·모건스탠리증권 수석 외환전략가는 "심리적 저항선으로 볼 수 있는 155엔을 넘어섰기에 단기적으로는 엔화 매수세가 유입되기 쉽다"고 닛케이에 전했다. 그러면서 "당분간은 올해 고점인 152엔대까지 상승 여지가 있을 것"이라고 덧붙였다.
'''
inference(news)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'stock_related: True\n\nsummary:\n일본 엔화가 달러당 153엔대까지 상승하며 반년 만에 최강세를 기록했다. 이번엔화 강세는 미국과 일본 통화당국이 추가 외환시장 개입 가능성과, 일본은행(BOJ)의 금리 인상 가속화 전망 등이 복합적으로 작용한 것으로 보인다.\n\npositive_stocks:\n- 일본은행(BOJ): 금리 인상 전망이 엔화 매수세를 부추기고 있다.\n- 일본 정부: 외환시장 개입 가능성 증가로 엔화 가치 상승에 긍정적 영향.\n\npositive_keywords:\n- 외환시장 개입 가능성\n- 금리 인상 전망\n- 안전자산 선호 현상 완화\n\npositive_reasons:\n외환시장 개입 가능성과 금리 인상 전망은 엔화에 대한 수요를 높이고, 안전자산 선호 현상이 완화되면서 매수세가 지속될 가능성이 높아졌다. 특히 BOJ의 금리 인상 속도 가속화는 엔화에 대한 투자 유인을 강화할 것이다.\n\nnegative_stocks:\n- 엔저: 엔화 강세로 주가가 하락할 가능성이 있다.\n- 달러 관련 자산: 엔화 강세로 달러 관련 자산의 가치가 하락할 수 있다.\n\nnegative_keywords:\n- 엔화 강세로 인한 엔저 하락\n- 달러 매도 압력\n\nnegative_reasons:\n엔화가 강세를 보이면서 엔저에 투자하던 투자자들이 포지션을 청산하고 달러로 전환하는 추세가 나타나고 있다. 이는 엔저의 가치를 낮추고, 달러 매도 압력을 가중시킬 수 있다.\n\n추가적으로, 달러당 155엔 선이 무너지면서 일부 대규모 손절매 주문이 실행되었고, 옵션 딜러들도 달러 매도에 나서 상승세에 힘을 실었다는 점도 주의할 필요가 있다. 하지만 장기적으로는 미국과 일본 통화당국이 엔화 약세를 정정하려는 구체적인 조치가 취해지지 않는 한, 엔화 강세는 지속될 가능성이 높다.'

In [35]:
news = ''' 
국고채 금리, 유가 상승에 낙폭 되돌리며 상승 마감(종합)

(서울=연합뉴스) 강수지 기자 = 8일 국고채 금리는 장중 하락분을 반납하고 소폭 상승 마감했다.

외국인 국채선물 순매수에도 미국과 이란 간 교전이 재개되면서 국제유가가 배럴당 100달러에 육박한 영향을 받았다.

이날 서울 채권시장에서 3년 만기 국고채 금리는 전 거래일보다 0.1bp(1bp=0.01%포인트) 오른 연 3.901%에 장을 마쳤다.

10년물 금리는 연 4.401%로 1.6bp 상승했다. 5년물과 2년물은 각각 0.7bp, 0.1bp 상승해 연 4.127%, 연 3.721%에 마감했다.

20년물은 연 4.560%로 0.7bp 내렸다. 30년물과 50년물은 각각 0.4bp, 0.3bp 상승해 연 4.635%, 연 4.543%를 기록했다.

이날 3년 국채선물은 전일 대비 1틱 하락한 103.11에, 10년 국채선물은 4틱 하락한 105.13에 거래를 마쳤다. 외국인 순매수에 장중 강세폭을 확대했으나, 지정학적 위기로 인한 유가 우려에 상승분을 반납했다.

외국인은 3년 선물을 4천556계약, 10년 선물을 1천376계약 순매수했다.

간밤 미국 금융시장이 노동절로 휴장한 가운데 국내 채권시장은 외국인 국채선물 매수에 주목하며 장중 강세를 나타냈다.

그러나 유럽시장 개장 무렵 국제유가 벤치마크인 브렌트유가 배럴당 100달러에 육박하면서 국내 채권시장은 강세를 반납했다. 브렌트유 선물 가격은 이날 오후 1.4%가량 오른 배럴당 98.65달러 수준에서 거래되고 있다.

이날 발표된 한국의 2분기 실질 국내총생산(GDP) 성장률 잠정치는 전분기 대비 0.6%, 전년 대비 3.7%로 속보치와 동일했다. 명목 GDP는 전기 대비 9.2%, 전년 대비 26.4% 성장했다.

증권사의 한 채권 중개인은 "이날 채권은 강세를 보였는데, 미국과 이란 충돌 우려에 유가가 오르면서 강세를 되돌렸다"며 "유가와 미국 금리 움직임이 중요한 가운데 다음주 있을 국채선물 롤오버(월물 교체)에서 외국인 움직임이 중요할 듯하다"고 말했다.
'''
inference(news)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'stock_related: True\n\nsummary:\n국내 채권시장은 외국인 국채선물 매수에 힘입어 장중 강세를 보였으나, 미국과 이란 간 교전 재개로 인한 국제유가 상승 우려로 인해 하락 분을 반납하고 소폭 상승 마감했다. 3년, 10년 국고채 금리는 각각 0.1%, 1.6% 상승했으며, 20년물은 0.7% 내렸지만 30년과 50년물은 각각 0.4%, 0.3% 상승했다.\n\npositive_stocks:\n- 외국인 순매수가 주된 원동력이었던 3년, 10년 국채선물에 대한 직접적인 영향을 받는 채권 관련 종목들이 긍정적일 수 있다.\n\npositive_keywords:\n- 외국인 순매수\n- 국채선물 롤오버 (월물 교체)\n\npositive_reasons:\n- 외국인 순매수가 채권시장에 지속적인 강세를 주입할 수 있기 때문에, 외국인 투자에 의존도가 높은 채권 관련 종목들은 긍정적 영향을 받을 수 있다. \n- 다음주 국채선물 롤오버 시 외국인의 투자 전략과 움직임이 주가에 영향을 미칠 수 있다.\n\nnegative_stocks:\n- 유가 상승에 따른 금리 상승 압력이 채권시장에 부정적 영향을 미칠 수 있는 종목들\n\nnegative_keywords:\n- 국제유가 상승\n- 지정학적 위기\n\nnegative_reasons:\n- 국제유가 상승으로 인해 지정학적 위기가 가중되면, 채권시장의 강세를 반납하게 되고 금리 상승 압력이 커질 수 있다. 이는 금융시스템에 부담을 주어 채권 관련 종목들의 가치를 하락시킬 수 있다.\n\n이 뉴스에 직접적으로 영향을 받는 특정 상장 종목을 명확히 지목하기는 어렵지만, 금리와 채권시장 동향이 주식시장 전반에 미치는 영향은 상당할 수 있다. 특히 금융, 부동산, 소비재 등과 같은 부채 비율이 높은 산업에 더 큰 영향을 줄 수 있다.'

# Base 모델과 비교

In [36]:
from transformers import AutoModelForCausalLM, pipeline

base_model_id = 'NCSOFT/Llama-VARCO-8B-Instruct' # 사전 학습 모델명

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    dtype = torch.bfloat16, # 가중치 로딩 dtype(bf16)
    device_map= 'auto'      # 환경에 맞춰 CPU / GPU 자동 배치
)

base_pipe = pipeline('text-generation', model=finetuned_model,tokenizer=tokenizer)

# 베이스 모델과 LoRA 파인튜닝 모델의 응답과 정답 비교
for idx, (prompt, label) in enumerate(zip(prompt_list[10:13], label_list[10:13])):
    print(f"[샘플 {idx +1}")
    base_resp = test_inference(base_pipe, prompt)
    lora_resp = test_inference(pipe, prompt)
    print(f"[Base - 파인튜닝 전] {base_resp}")
    print(f"[LoRA - 파인튜닝 후] {lora_resp}")
    print(f"[Label] {label}")
    print("=" * 100)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[샘플 1


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Base - 파인튜닝 전] Summary:

글로벌 경제는 2023년에도 지속적인 침체로 접어들 것으로 전망되며, 미국의 연방준비제도(Fed)는 인플레이션을 억제하기 위해 추가적인 금리 인상 전망이다. 이는 세계 각국의 경제성장 둔화와 함께 국제통화기금(IMF)의 세계 경제 성장률 전망을 낮추는 요인이다. 

Stock-related:

stock_related: True

Positive Stocks:

positive_stocks: 
- 
  stock: 
    - "AAPL" (Apple Inc.)
    - "MSFT" (Microsoft Corporation)
    - "AMZN" (Amazon.com Inc.)
    - "GOOGL" (Alphabet Inc.)
    - "V" (Visa Inc.)
  positive_keywords: 
    - "디지털화"
    - "클라우드 컴퓨팅"
    - "온라인 쇼핑"
    - "데이터 분석"
    - "디지털 결제"
  positive_reasons:
    - "글로벌 경제 침체에도 불구하고, 위의 기업들은 디지털화와 클라우드 컴퓨팅, 온라인 쇼핑 등에 대한 수요가 상대적으로 높아질 가능성이 있다. 특히, 'AAPL'의 Apple은 개인용 컴퓨터와 관련 제품의 수요가 상대적으로 안정적일 수 있으며, 'MSFT'의 Microsoft는 기업 클라우드 솔루션에 대한 수요가 증가할 수 있다. 'AMZN'의 Amazon은 온라인 쇼핑에 대한 수요가 지속될 것이고, 'GOOGL'의 Alphabet(구글)은 데이터 분석 및 디지털 마케팅 서비스에 대한 수요가 높아질 수 있다. 'V'의 Visa는 디지털 결제의 확산으로 인해 수요가 증가할 수 있다."

Negative Stocks:

negative_stocks:
- 
  stock: 
    - "NIO" (NIO Inc.)
    - "LYFT" (Lyft, Inc.)
    - "ZM" (Zoom Video Communications, Inc.

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Base - 파인튜닝 전] Summary:

글로벌 경제는 2023년에도 지속적인 침체로 접어들 것으로 전망되며, 미국의 연방준비제도(Fed)는 인플레이션을 억제하기 위해 추가적인 금리 인상 전망이다. 이는 세계 각국의 경제성장 둔화와 함께 국제통화기금(IMF)의 세계 경제 성장률 전망을 낮추는 요인이다. 

Stock-related:

stock_related: True

Positive Stocks:

positive_stocks: 
- 
  stock: 
    - "AAPL" (Apple Inc.)
    - "MSFT" (Microsoft Corporation)
    - "AMZN" (Amazon.com Inc.)
    - "GOOGL" (Alphabet Inc.)
    - "V" (Visa Inc.)
  positive_keywords: 
    - "디지털화"
    - "클라우드 컴퓨팅"
    - "온라인 쇼핑"
    - "데이터 분석"
    - "디지털 결제"
  positive_reasons:
    - "글로벌 경제 침체에도 불구하고, 위의 기업들은 디지털화와 클라우드 컴퓨팅, 온라인 쇼핑 등에 대한 수요가 상대적으로 높아질 가능성이 있다. 특히, 'AAPL'의 Apple은 개인용 컴퓨터와 관련 제품의 수요가 상대적으로 안정적일 수 있으며, 'MSFT'의 Microsoft는 기업 클라우드 솔루션에 대한 수요가 증가할 수 있다. 'AMZN'의 Amazon은 온라인 쇼핑에 대한 수요가 지속될 것이고, 'GOOGL'의 Alphabet(구글)은 데이터 분석 및 디지털 마케팅 서비스에 대한 수요가 높아질 수 있다. 'V'의 Visa는 디지털 결제의 확산으로 인해 수요가 증가할 수 있다."

Negative Stocks:

negative_stocks:
- 
  stock: 
    - "NIO" (NIO Inc.)
    - "LYFT" (Lyft, Inc.)
    - "ZM" (Zoom Video Communications, Inc.

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Base - 파인튜닝 전] Summary:

글로벌 경제는 2023년에도 지속적인 침체로 접어들 것으로 전망되며, 미국의 연방준비제도(Fed)는 인플레이션을 억제하기 위해 추가적인 금리 인상 전망이다. 이는 세계 각국의 경제성장 둔화와 함께 국제통화기금(IMF)의 세계 경제 성장률 전망을 낮추는 요인이다. 

Stock-related:

stock_related: True

Positive Stocks:

positive_stocks: 
- 
  stock: 
    - "AAPL" (Apple Inc.)
    - "MSFT" (Microsoft Corporation)
    - "AMZN" (Amazon.com Inc.)
    - "GOOGL" (Alphabet Inc.)
    - "V" (Visa Inc.)
  positive_keywords: 
    - "디지털화"
    - "클라우드 컴퓨팅"
    - "온라인 쇼핑"
    - "데이터 분석"
    - "디지털 결제"
  positive_reasons:
    - "글로벌 경제 침체에도 불구하고, 위의 기업들은 디지털화와 클라우드 컴퓨팅, 온라인 쇼핑 등에 대한 수요가 상대적으로 높아질 가능성이 있다. 특히, 'AAPL'의 Apple은 개인용 컴퓨터와 관련 제품의 수요가 상대적으로 안정적일 수 있으며, 'MSFT'의 Microsoft는 기업 클라우드 솔루션에 대한 수요가 증가할 수 있다. 'AMZN'의 Amazon은 온라인 쇼핑에 대한 수요가 지속될 것이고, 'GOOGL'의 Alphabet(구글)은 데이터 분석 및 디지털 마케팅 서비스에 대한 수요가 높아질 수 있다. 'V'의 Visa는 디지털 결제의 확산으로 인해 수요가 증가할 수 있다."

Negative Stocks:

negative_stocks:
- 
  stock: 
    - "NIO" (NIO Inc.)
    - "LYFT" (Lyft, Inc.)
    - "ZM" (Zoom Video Communications, Inc.

In [37]:
# HuggingFace에 finetuning 후 자동으로 업로드 되지 않은 경우 수동으로 업로드
import os  # 환경변수(HF_TOKEN) 사용
from huggingface_hub import HfApi  # Hub API 사용

repo_id = "kkkfu/Llama-VARCO-8b-news2stock-analyzer"  # 업로드할 모델 repo id
local_dir = "./Llama-VARCO-8b-news2stock-analyzer"  # 로컬 모델 폴더 경로

api = HfApi(token=os.environ["HF_TOKEN"])  # Hub 인증 토큰으로 API 객체 생성
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)  # repo가 없으면 생성(있으면 그대로 사용)

api.upload_folder(  # 로컬 폴더 전체를 Hub로 업로드
    folder_path=local_dir,  # 업로드할 로컬 폴더
    repo_id=repo_id,  # 대상 repo
    repo_type="model",  # 모델 repo로 업로드
    ignore_patterns=["checkpoint-*", "**/checkpoint-*"],  # 체크포인트 폴더는 제외
    commit_message="Upload final LoRA adapter (without checkpoints)"  # 커밋 메시지
)

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/kkkfu/Llama-VARCO-8b-news2stock-analyzer/commit/91bba208d7ee68bef937bf408e5f0859886b8238', commit_message='Upload final LoRA adapter (without checkpoints)', commit_description='', oid='91bba208d7ee68bef937bf408e5f0859886b8238', pr_url=None, repo_url=RepoUrl('https://huggingface.co/kkkfu/Llama-VARCO-8b-news2stock-analyzer', endpoint='https://huggingface.co', repo_type='model', repo_id='kkkfu/Llama-VARCO-8b-news2stock-analyzer'), pr_revision=None, pr_num=None)